# 03 — Live experiment against the deployed MSK cluster

Unlike `00`-`02`, which point at `$FDAI_TARGET` (local by default), this one is
pinned to the real cloud cluster on purpose — there is no ambiguity about which
broker you are looking at. A few small, quick experiments on whatever is
actually flowing right now, not a full analysis (see `01_explore_trades.ipynb`
for that).

**Prerequisites:** `make up` has succeeded, and this environment has the same
AWS credentials active that `terraform` used — `from_terraform()` shells out to
`terraform output` under the hood.

## If AWS credentials have expired

Sandbox SSO sessions commonly expire in hours — well before the account's
7-day wipe. If the next cell raises `TargetError: AWS credentials are
invalid or expired...`, run this first, then re-run it — no need to
restart Jupyter (see `notebooks/README.md` Troubleshooting).

`fdai-sandbox` is the example profile name from `docs/SETUP.md` §2;
replace it if yours differs. Skip this cell entirely if your org issues
raw/temporary keys instead of SSO.

In [1]:
!aws sso login --profile fdai-sandbox

Attempting to open your default browser. If the browser does not open, open the following URL.
If you are unable to open the URL on this device, run this command again with the '--use-device-code' option.

https://oidc.eu-central-1.amazonaws.com/authorize?response_type=code&client_id=a0YtZYShATkIdsWaPYtBNWV1LWNlbnRyYWwtMQ&redirect_uri=http%3A%2F%2F127.0.0.1%3A56427%2Foauth%2Fcallback&state=1bfcdd44-85af-4ff1-aa40-c8bea4e06097&code_challenge_method=S256&scopes=sso%3Aaccount%3Aaccess&code_challenge=U-qN-uOVz6VjaloDc8eZZP6wcGe_c6RdX0TfW9BPogw

Successfully logged into Start URL: https://identitycenter.amazonaws.com/ssoins-69871294d6700676


In [2]:
%env AWS_PROFILE=fdai-sandbox

env: AWS_PROFILE=fdai-sandbox


In [3]:
import devlab
from devlab import frames

target = devlab.from_terraform()  # the deployed MSK cluster, explicitly — not $FDAI_TARGET
target

TargetError: `terraform -chdir=/Users/lamnguyen/Documents/finance_data_ai/infra/envs/dev output -raw bootstrap_brokers_public` failed. Is the stack up? Run `make up` first.
[31m╷[0m[0m
[31m│[0m [0m[1m[31mError: [0m[0m[1mFailed to load state: S3 bucket "fdai-tfstate-160071257600" does not exist.
[31m│[0m [0m
[31m│[0m [0mThe referenced S3 bucket must have been previously created. If the S3 bucket
[31m│[0m [0mwas created within the last minute, please wait for a minute or two and try
[31m│[0m [0magain.
[31m│[0m [0m
[31m│[0m [0mError: operation error S3: ListObjectsV2, https response error StatusCode: 404, RequestID: RP014HDY8GX01XJT, HostID: 9lihuIXH1Y/E7OsFpm0QokHG1TzNGuZ1FVRh/0oKmzMTG9ZdkiG28eEyiXLDendKx8Y6M1D3enw=, NoSuchBucket: 
[31m│[0m [0m[0m
[31m│[0m [0m
[31m│[0m [0m[0m
[31m╵[0m[0m

## Is it actually live right now

Cheap, and worth doing before anything else: reads from `latest` for 10
seconds. Zero here means the producer host isn't running — check that before
assuming anything below is broken.

In [9]:
report = devlab.rate(target, seconds=10.0)
print(f"{report.messages} trades in {report.seconds:.1f}s = {report.per_second:.1f}/s")
report.by_venue

NameError: name 'target' is not defined

## Grab a batch of real trades

Bounded by both a count and a clock, so this returns even against a quiet
topic. `offset_reset="earliest"` reads what's already retained rather than
only what arrives from this point on.

In [ ]:
records = devlab.collect(target, limit=2_000, seconds=30.0, offset_reset="earliest")
df = frames.trades_frame(records)
print(f"{len(df):,} trades  {df['event_ts'].min()} .. {df['event_ts'].max()}")
df.head()

## Experiment 1 — who's trading what

Per venue and instrument: trade count, volume, and notional. Sorted by
notional, so whatever's actually moving money floats to the top.

In [ ]:
(
    df.groupby(["venue", "instrument_id"], observed=True)
    .agg(trades=("trade_id", "count"), volume=("size", "sum"), notional=("notional", "sum"))
    .sort_values("notional", ascending=False)
)

## Experiment 2 — price, live

Whichever instrument traded the most in this window, plotted as-is (no
resampling) — the rawest possible look at the tape.

In [ ]:
top_instrument = df["instrument_id"].value_counts().idxmax()
subset = df[df["instrument_id"] == top_instrument]
axis = subset.plot(x="event_ts", y="price", figsize=(10, 3), title=f"{top_instrument} — live")
axis.set_xlabel("event time (UTC)")

## Experiment 3 — a quick VWAP

Dedupe first (a repaired trade can appear twice — see `02_prototype_silver.ipynb`
for why), then 1-minute bars. Just the tail, since this is meant to be a quick
look, not the full analysis.

In [ ]:
bars = frames.bars(frames.dedupe(df), freq="1min")
bars.tail()